<a href="https://colab.research.google.com/github/nayely77/TIC-ASR-Streaming-Accesibilidad/blob/main/notebooks/whisper_colab_rapido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Whisper-small español — versión rápida (Common Voice + LoRA)

Optimizada para terminar en **~2–3 h totales** en la T4 gratuita de Colab:

| Celda | Paso | Tiempo |
|---|---|---|
| 1 | GPU T4 | — |
| 2 | Instalar dependencias | ~2 min |
| 3 | Montar Drive | ~1 min |
| 4 | Escribir el script | <1 s |
| 5 | Datos: **30 h** train + 0.5 h validación | ~10–20 min |
| 6 | Entrenar (3 épocas, batch 16) | ~2 h |
| 7 | Probar el modelo | ~1 min |

**Qué cambia frente a la versión de 40 h** (y por qué no perjudica el resultado):
- **30 h en vez de 40 h** → ~25 % menos pasos; diferencia de WER típica ≈ 1 punto.
- **Sin gradient checkpointing** → ~1.3× más rápido; el entrenamiento es *matemáticamente idéntico* (solo usa más VRAM, y en 16 GB sobra).
- **Validación de 0.5 h y eval cada 750 pasos** → menos tiempo perdido en evaluar; el WER medido sigue siendo fiable.

> Si algún día te quedas sin VRAM (`CUDA out of memory`): añade `--grad_checkpointing`
> y, si persiste, `--batch_size 8 --grad_accum 2` en la celda 6.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "Sin GPU. Ve a: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU y vuelve a ejecutar."
)
print("\n[GPU]", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB de VRAM")

## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers "datasets[audio]" accelerate peft evaluate jiwer soundfile librosa python-dotenv

import transformers, datasets, peft, torch
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("peft        ", peft.__version__)
print("torch       ", torch.__version__)

## 2. Montar Google Drive

Los checkpoints y el modelo final quedan a salvo aunque se desconecte la sesión.
No se pide token de HF: el espejo público `fixie-ai/common_voice_17_0` no lo necesita.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_BASE = "/content/drive/MyDrive/whisper_es"
except Exception as e:
    OUT_BASE = "/content/whisper_es"
    print("Drive no disponible, se usará disco efímero:", e)

os.makedirs(OUT_BASE, exist_ok=True)
print("Checkpoints y modelo final se guardarán en:", OUT_BASE)

## 3. Escribir el script de fine-tuning

In [ ]:
%%writefile whisper_finetune.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fine-tuning de Whisper-small en español (Common Voice) con LoRA (PEFT).
Diseñado para una GPU de 6 GB (RTX 3060 Laptop) con CUDA 12.4.

Qué hace
--------
1) Descarga Common Voice (config "es") por shards parquet y materializa en disco
   las horas de audio que pidas en FLAC 16 kHz: RAM acotada y progreso REANUDABLE
   (manifiesto JSONL: si la sesión muere, vuelve a ejecutar la misma celda y
   continúa donde quedó). Caché compartida con wav2vec2_finetune.py.
2) Entrena Whisper-small con adaptadores LoRA + fp16 + gradient checkpointing,
   evaluando el WER cada N pasos sobre ~1 h de validación.
3) Fusiona los adaptadores LoRA dentro de los pesos (merge_and_unload) y guarda
   el modelo final en --output_dir, listo para usarse en otro proyecto con
   transformers.pipeline().

Uso
---
    export HF_TOKEN=hf_xxx            # cuenta HF con Common Voice aceptado
    python whisper_finetune.py --prepare_data_only   # 1) solo descargar datos
    python whisper_finetune.py                        # 2) entrenar y exportar
    python whisper_finetune.py --resume               #    continuar checkpoint
"""

import argparse
import gc
import json
import os
import re
import shutil
import sys
import time
from dataclasses import dataclass
from typing import Any, Dict, List
from dotenv import load_dotenv

import numpy as np

# ------------------------------------------------------------------ texto --
# Letras permitidas (minúsculas, con acentos y ñ), dígitos y espacio.
_ALLOWED = "abcdefghijklmnopqrstuvwxyz0123456789áéíóúüñ "
_NOT_ALLOWED_RE = re.compile(r"[^" + _ALLOWED + "]")


load_dotenv()

# Descarga clásica por HTTP en vez de xet: hf_xet viene activado por defecto en
# huggingface_hub>=1.0 (los Colab recientes) y puede acumular varios GB de RAM
# reconstruyendo archivos grandes. Por el CDN clásico el consumo es mínimo.
# setdefault = si quieres xet, exporta HF_HUB_DISABLE_XET=0 antes de ejecutar.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

def clean_text_es(text: str) -> str:
    """Normaliza una transcripción: minúsculas, sin puntuación, espacios simples."""
    text = _NOT_ALLOWED_RE.sub(" ", (text or "").lower())
    return re.sub(r"\s+", " ", text).strip()


# -------------------------------------------------------------------- cli --
def parse_args():
    p = argparse.ArgumentParser(
        description="Fine-tuning LoRA de Whisper para ASR en español (Common Voice).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    datos = p.add_argument_group("datos")
    #datos.add_argument("--dataset_name", default="mozilla-foundation/common_voice_17_0")
    datos.add_argument("--dataset_name", default="fixie-ai/common_voice_17_0")
    datos.add_argument("--dataset_config", default="es")
    datos.add_argument("--max_hours", type=float, default=40.0,
                       help="horas de audio de entrenamiento a materializar (<=0: sin límite)")
    datos.add_argument("--eval_hours", type=float, default=1.0,
                       help="horas de validación para medir el WER")
    datos.add_argument("--data_cache", default="./data/cv_es",
                       help="caché de audio/datasets (compartida con wav2vec2_finetune.py)")
    datos.add_argument("--shuffle_buffer", type=int, default=2048,
                       help="tamaño del buffer de shuffle en modo streaming")
    datos.add_argument("--no_streaming", action="store_true",
                       help="descargar el dataset completo (mucho más disco) en vez de streaming")
    datos.add_argument("--prepare_data_only", action="store_true",
                       help="solo materializar los datos y salir")
    datos.add_argument("--ram_cap_gb", type=float, default=0.0,
                       help="límite de RAM (GB) al materializar; 0 = automático "
                            "(75%% de la RAM instalada)")

    modelo = p.add_argument_group("modelo")
    modelo.add_argument("--model_id", default="openai/whisper-small")
    modelo.add_argument("--language", default="es",
                        help="código de idioma para el tokenizador de Whisper")
    modelo.add_argument("--output_dir", default="./models/whisper-small-es-lora",
                        help="carpeta del modelo final fusionado")
    modelo.add_argument("--trainer_dir", default="./runs/whisper_small_es",
                        help="carpeta de checkpoints del Trainer")

    lora = p.add_argument_group("lora")
    lora.add_argument("--lora_r", type=int, default=32)
    lora.add_argument("--lora_alpha", type=int, default=64)
    lora.add_argument("--lora_dropout", type=float, default=0.05)

    fit = p.add_argument_group("entrenamiento")
    fit.add_argument("--epochs", type=float, default=3.0)
    fit.add_argument("--batch_size", type=int, default=8,
                     help="si se queda sin VRAM, baja a 4 y sube --grad_accum")
    fit.add_argument("--eval_batch_size", type=int, default=8)
    fit.add_argument("--grad_accum", type=int, default=2,
                     help="batch efectivo = batch_size * grad_accum")
    fit.add_argument("--lr", type=float, default=5e-4,
                     help="tasa de aprendizaje de los adaptadores LoRA")
    fit.add_argument("--warmup_steps", type=int, default=100)
    fit.add_argument("--weight_decay", type=float, default=0.01)
    fit.add_argument("--logging_steps", type=int, default=50)
    fit.add_argument("--eval_steps", type=int, default=500)
    fit.add_argument("--save_steps", type=int, default=500)
    fit.add_argument("--num_workers", type=int, default=4,
                     help="procesos del dataloader (extracción log-mel en paralelo)")
    fit.add_argument("--seed", type=int, default=42)
    fit.add_argument("--bf16", action="store_true",
                     help="usar bfloat16 en lugar de float16")
    fit.add_argument("--grad_checkpointing", action="store_true",
                     help="activa gradient checkpointing (ahorra VRAM, ~1.3x más lento); "
                          "actívalo solo si te quedas sin VRAM")
    fit.add_argument("--resume", action="store_true",
                     help="continuar desde el último checkpoint de --trainer_dir")
    return p.parse_args()


# ------------------------------------------------------------------ datos --
_TOKEN_WARNED = False


def get_hf_token():
    """Devuelve el token de HF si existe (sin exigirlo).

    El espejo fixie-ai/common_voice_17_0 es público: el token solo hace falta
    para el repo oficial gated (mozilla-foundation/...). Nunca imprimas el token.
    """
    global _TOKEN_WARNED
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from huggingface_hub import get_token
            token = get_token()
        except Exception:
            token = None
    if not token and not _TOKEN_WARNED:
        _TOKEN_WARNED = True
        print(
            "[aviso] Sin token de HF: solo se pueden cargar repos públicos "
            "(p. ej. fixie-ai/common_voice_17_0). Para el repo oficial "
            "(mozilla-foundation/...) exporta HF_TOKEN con una cuenta que haya "
            "aceptado los términos del dataset."
        )
    return token


def _ram_gb() -> float:
    """RAM residente del proceso en GB (-1 si psutil no está disponible)."""
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**30
    except Exception:  # noqa: BLE001
        return -1.0


class _RamCap(Exception):
    """Señal interna: se alcanzó el límite de RAM durante la materialización."""


def _ram_cap_gb(args) -> float:
    """Límite de RAM para materializar: --ram_cap_gb o 75% de la RAM instalada."""
    if args.ram_cap_gb > 0:
        return args.ram_cap_gb
    try:
        import psutil
        return psutil.virtual_memory().total / 2**30 * 0.75
    except Exception:  # noqa: BLE001
        return 0.0  # sin límite conocible: no hay vigilante


def _check_ram(cap_gb: float, n: int, total_s: float) -> None:
    """Corta LA MATERIALIZACIÓN (no la sesión) si la RAM supera el límite."""
    ram = _ram_gb()
    if cap_gb and ram > cap_gb:
        raise _RamCap(f"RAM {ram:.1f} GB > límite {cap_gb:.1f} GB "
                      f"(llevabas {n} clips, {total_s / 3600:.2f} h)")


def _manifest_file(args, split: str) -> str:
    return os.path.join(args.data_cache, f"manifest_{split}.jsonl")


def _load_manifest(args, split: str):
    """Clips ya convertidos según el manifiesto (reanudación tras una caída).

    Descarta entradas cuyo FLAC ya no exista y reescribe el manifiesto limpio.
    Devuelve [(path, sentence, dur), ...].
    """
    mf = _manifest_file(args, split)
    if not os.path.isfile(mf):
        return []
    rows = []
    with open(mf, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                if os.path.isfile(d["path"]):
                    rows.append((d["path"], d["sentence"], float(d["duration"])))
            except Exception:  # noqa: BLE001
                continue
    tmp = mf + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for p, s, d in rows:
            f.write(json.dumps({"path": p, "sentence": s, "duration": d}) + "\n")
    os.replace(tmp, mf)
    return rows


def _download_shard(args, fname: str, shard_dir: str) -> str:
    """hf_hub_download con reintentos: el Hub responde 429 a IPs compartidas
    (Colab incluida) y sin reintentos la celda moriría a mitad de la descarga."""
    from huggingface_hub import hf_hub_download

    last = None
    for intento, espera in enumerate((15, 45, 90), start=1):
        try:
            return hf_hub_download(args.dataset_name, fname, repo_type="dataset",
                                   token=get_hf_token(), local_dir=shard_dir)
        except Exception as e:  # noqa: BLE001
            last = e
            print(f"[datos] Descarga de {os.path.basename(fname)} falló "
                  f"(intento {intento}/3): {type(e).__name__}; reintento en "
                  f"{espera} s ...", flush=True)
            time.sleep(espera)
    raise last


def _iter_dataset(args, split: str, light: bool = False):
    """Carga un split de Common Voice como audio 16 kHz mono (streaming por defecto).

    light=True (materialización): usa un buffer de mezcla más pequeño. El orden
    por época ya lo decide el RandomSampler del Trainer sobre el dataset
    materializado, así que aquí basta una pre-mezcla ligera y ahorramos RAM.
    """
    from datasets import Audio, load_dataset

    streaming = not args.no_streaming
    try:
        ds = load_dataset(
            args.dataset_name,
            args.dataset_config,
            split=split,
            streaming=streaming,
            token=get_hf_token(),
        )
    except Exception as e:  # noqa: BLE001
        sys.exit(
            f"\n[ERROR] No se pudo cargar {args.dataset_name}/{args.dataset_config} ({split}): {e}\n"
            "Revisa que el token sea válido, que hayas aceptado los términos del dataset\n"
            "y que 'datasets' esté en la versión indicada en requirements.txt.\n"
        )
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    if split == "train":
        if streaming:
            # El buffer de mezcla PRECARGA ejemplos antes de devolver el primero:
            # parece un cuelgue y consume RAM proporcional a su tamaño. En modo
            # ligero lo capamos a 1024; el orden por época lo decide después el
            # RandomSampler del Trainer sobre el dataset ya materializado.
            buffer_n = min(args.shuffle_buffer, 1024) if light else args.shuffle_buffer
            print(f"[datos] Llenando buffer de mezcla ({buffer_n} clips): "
                  "puede tardar unos minutos sin salida en pantalla...", flush=True)
            ds = ds.shuffle(buffer_size=buffer_n, seed=args.seed)
        else:
            ds = ds.shuffle(seed=args.seed)
    return ds


def _prepare_dirs(args, split: str):
    """Prepara las carpetas de caché y limpia materializaciones a medias."""
    out_dir = os.path.join(args.data_cache, split)
    audio_dir = os.path.join(args.data_cache, "audio")
    os.makedirs(audio_dir, exist_ok=True)
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    return out_dir, audio_dir


def _finish_materialization(rows, n, total_s, split, out_dir, t0):
    """Guarda el dataset de rutas + metadatos con el formato de 'datasets'."""
    if n == 0:
        sys.exit("[ERROR] No se materializó ningún clip. Revisa token/términos del dataset.")
    from datasets import Audio, Dataset

    tmp = out_dir + "_tmp"  # guardado atómico: si muere a mitad no queda a medias
    if os.path.isdir(tmp):
        shutil.rmtree(tmp)
    Dataset.from_dict(
        {
            "audio": [r[0] for r in rows],
            "sentence": [r[1] for r in rows],
            "duration": [r[2] for r in rows],
        }
    ).cast_column("audio", Audio(sampling_rate=16000, decode=False)).save_to_disk(tmp)
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.replace(tmp, out_dir)
    print(f"[datos] '{split}' listo: {n} clips | {total_s/3600:.2f} h -> {out_dir} "
          f"({time.time()-t0:.0f} s)\n", flush=True)


def _list_parquet_shards(args, split: str):
    """Lista los shards parquet de un split del repo (p. ej. es/train-*.parquet).

    Con reintentos: el Hub responde 429 (rate limit) a menudo a IPs compartidas
    como las de Colab; sin reintentos degradaríamos silenciosamente al modo
    streaming, que es justo el que dispara el consumo de RAM.
    """
    from huggingface_hub import HfApi

    last = None
    for intento, espera in enumerate((10, 30, 60), start=1):
        try:
            api = HfApi(token=get_hf_token())
            files = api.list_repo_files(args.dataset_name, repo_type="dataset")
            break
        except Exception as e:  # noqa: BLE001
            last = e
            print(f"[datos] Listado de shards falló (intento {intento}/3): "
                  f"{type(e).__name__}; reintento en {espera} s ...", flush=True)
            time.sleep(espera)
    else:
        raise last

    cfg_pat = re.compile(rf"(?:^|/){re.escape(args.dataset_config)}/")
    shards = sorted(
        f for f in files
        if f.endswith(".parquet") and cfg_pat.search(f)
        and (os.path.basename(f).startswith(f"{split}-")
             or f.split("/")[-2] == split)
    )
    return shards


def _resample_to_16k(arr, sr: int):
    """Re-muestrea a 16 kHz (soxr si está; si no, librosa)."""
    try:
        import soxr
        return soxr.resample(arr, sr, 16000, quality="HQ")
    except Exception:  # noqa: BLE001
        import librosa
        return librosa.resample(arr, orig_sr=sr, target_sr=16000)


def _materialize_parquet(args, split: str, max_hours: float, shards, cap_gb: float):
    """Vía directa: descarga shards parquet UNO A UNO y los convierte a FLAC 16 kHz.

    RAM acotada (~1 GB) sin importar las horas:
    - cada shard se lee por row groups (~100 clips) y en 1 solo hilo;
    - cada clip se escribe a FLAC **y** a un manifiesto JSONL al momento: el
      audio decodificado nunca acumula en RAM (vive un clip y se libera);
    - borra cada shard al terminar y devuelve al SO la memoria que el pool de
      pyarrow retiene (release_unused). NO usa el streaming de 'datasets', que
      en Colab free consumió toda la RAM (fugas de fsspec/pyarrow).
    - vigilante de RAM (_check_ram): si RSS supera --ram_cap_gb (o el 75% del
      total), corta EN ORDEN conservando el progreso; al re-ejecutar la celda
      se reanuda desde el último clip (manifiesto), sin perder la descarga.
    """
    import io
    import random

    import pyarrow as pa
    import pyarrow.parquet as pq
    import soundfile as sf

    out_dir, audio_dir = _prepare_dirs(args, split)
    shard_dir = os.path.join(args.data_cache, "shards")
    os.makedirs(shard_dir, exist_ok=True)

    shards = list(shards)
    # aleatoriza QUÉ shards forman el subconjunto (equivalente al shuffle
    # del modo streaming, pero sin mantener nada en RAM)
    random.Random(args.seed).shuffle(shards)

    rows = _load_manifest(args, split)
    n, total_s = len(rows), sum(r[2] for r in rows)
    if rows:
        print(f"[datos] Reanudando: {n} clips ({total_s/3600:.2f} h) ya convertidos; "
              "sigo desde ahí.", flush=True)

    print(f"[datos] Modo directo por shards (RAM acotada): {len(shards)} parquet "
          f"de '{split}' en {args.dataset_name}")
    print(f"[datos] Materializando {max_hours or 'todo'} h (suele tardar 10-25 min): "
          "NO interrumpas la celda.", flush=True)
    limit_s = (max_hours or 0) * 3600.0
    t0 = time.time()
    mf = open(_manifest_file(args, split), "a", encoding="utf-8")

    status = "completo"
    try:
        hechos = 0  # shards procesados hasta ahora
        for si, fname in enumerate(shards):
            if limit_s and total_s >= limit_s:
                break
            _check_ram(cap_gb, n, total_s)
            local = _download_shard(args, fname, shard_dir)
            pf = pq.ParquetFile(local, memory_map=False, buffer_size=1 << 20)
            for batch in pf.iter_batches(batch_size=64, columns=["audio", "sentence"],
                                         use_threads=False):
                for item in batch.to_pylist():
                    if limit_s and total_s >= limit_s:
                        break
                    try:
                        blob = (item.get("audio") or {}).get("bytes")
                        sentence = (item.get("sentence") or "").strip()
                        if not blob or not sentence:
                            continue
                        arr, sr = sf.read(io.BytesIO(blob), dtype="float32")
                        if arr.ndim > 1:  # por si acaso, a mono
                            arr = arr.mean(axis=1)
                        if sr != 16000:
                            arr = _resample_to_16k(arr, sr)
                            sr = 16000
                        dur = arr.size / sr
                        if dur < 0.5 or dur > 30.0:  # Whisper usa clips de hasta 30 s
                            continue
                        path = os.path.join(audio_dir, f"{split}_{n:06d}.flac")
                        sf.write(path, arr, sr, format="FLAC", subtype="PCM_16")
                        rows.append((path, sentence, dur))
                        mf.write(json.dumps({"path": path, "sentence": sentence,
                                             "duration": dur}) + "\n")
                        total_s += dur
                        n += 1
                        if n == 1:
                            print("    [datos] Primer clip convertido: descarga y "
                                  "conversión en marcha...", flush=True)
                        if n % 500 == 0:
                            mf.flush()
                            ram = _ram_gb()
                            ram_txt = f" | RAM {ram:.1f} GB" if ram >= 0 else ""
                            print(f"    {n} clips | {total_s/3600:.2f} h | "
                                  f"{time.time()-t0:.0f} s{ram_txt}", flush=True)
                            gc.collect()
                            _check_ram(cap_gb, n, total_s)
                    except Exception:  # noqa: BLE001
                        continue  # clips corruptos o ilegibles: se omiten
                del batch
            del pf
            try:
                os.remove(local)  # el shard ya convertido no lo necesitamos en disco
            except OSError:
                pass
            gc.collect()
            try:
                pa.default_memory_pool().release_unused()
            except Exception:  # noqa: BLE001
                pass
            hechos += 1
            print(f"    [datos] shard {si+1}/{len(shards)} hecho | {total_s/3600:.2f} h "
                  "acumuladas", flush=True)
    except _RamCap as e:
        status = "parcial"
        print(f"\n[aviso] {e}.", flush=True)
        print("[aviso] Corto aquí EN ORDEN: conservo TODO lo ya convertido.", flush=True)
    except KeyboardInterrupt:
        status = "parcial"
        print("\n[aviso] Interrumpido: conservo TODO lo convertido hasta ahora.",
              flush=True)
    finally:
        mf.close()

    alcanzado = bool(limit_s) and total_s >= limit_s
    agotados = hechos >= len(shards)
    if status != "completo" and not (alcanzado or agotados):
        print(f"[datos] Progreso guardado: {n} clips | {total_s/3600:.2f} h "
              f"(objetivo {max_hours} h).", flush=True)
        print("[datos] Vuelve a ejecutar la MISMA celda para reanudar desde el "
              "último clip: no se pierde nada.", flush=True)
        return
    _finish_materialization(rows, n, total_s, split, out_dir, t0)


def _materialize_streaming(args, split: str, max_hours: float, cap_gb: float):
    """Respaldo: modo streaming de 'datasets' (puede consumir mucha RAM).

    Mismas protecciones que la vía por shards: cada clip va a FLAC + manifiesto
    al momento (nada acumula en RAM), vigilante de RAM y reanudación."""
    import soundfile as sf

    out_dir, audio_dir = _prepare_dirs(args, split)
    print(f"[datos] Materializando split '{split}' en modo STREAMING "
          f"(objetivo: {max_hours or 'todo'} h) ...")
    print("[datos] Suele tardar 15-30 min según la conexión: NO interrumpas la celda.")
    limit_s = (max_hours or 0) * 3600.0

    rows = _load_manifest(args, split)
    n, total_s = len(rows), sum(r[2] for r in rows)
    if rows:
        print(f"[datos] Reanudando: {n} clips ({total_s/3600:.2f} h) ya convertidos; "
              "sigo desde ahí.", flush=True)
    t0 = time.time()
    mf = open(_manifest_file(args, split), "a", encoding="utf-8")

    status = "completo"
    try:
        for item in _iter_dataset(args, split, light=True):
            try:
                audio = item["audio"]
                arr = np.asarray(audio["array"], dtype=np.float32)
                sentence = (item.get("sentence") or "").strip()
                if arr.size == 0 or not sentence:
                    continue
                dur = arr.size / audio["sampling_rate"]
                if dur < 0.5 or dur > 30.0:  # Whisper usa clips de hasta 30 s
                    continue
                path = os.path.join(audio_dir, f"{split}_{n:06d}.flac")
                sf.write(path, arr, audio["sampling_rate"], format="FLAC",
                         subtype="PCM_16")
                rows.append((path, sentence, dur))
                mf.write(json.dumps({"path": path, "sentence": sentence,
                                     "duration": dur}) + "\n")
                total_s += dur
                n += 1
                if n == 1:
                    print("    [datos] Primer clip recibido: descarga y conversión "
                          "en marcha...", flush=True)
                if n % 500 == 0:
                    mf.flush()
                    ram = _ram_gb()
                    ram_txt = f" | RAM {ram:.1f} GB" if ram >= 0 else ""
                    print(f"    {n} clips | {total_s/3600:.2f} h | "
                          f"{time.time()-t0:.0f} s{ram_txt}", flush=True)
                    gc.collect()
            except Exception:  # noqa: BLE001
                continue  # clips corruptos o ilegibles: se omiten
            if limit_s and total_s >= limit_s:
                break
            _check_ram(cap_gb, n, total_s)
    except _RamCap as e:
        status = "parcial"
        print(f"\n[aviso] {e}.", flush=True)
        print("[aviso] Corto aquí EN ORDEN: conservo TODO lo ya convertido.", flush=True)
    except KeyboardInterrupt:
        status = "parcial"
        print("\n[aviso] Interrumpido: conservo TODO lo convertido hasta ahora.",
              flush=True)
    finally:
        mf.close()

    alcanzado = bool(limit_s) and total_s >= limit_s
    if status != "completo" and not alcanzado:
        print(f"[datos] Progreso guardado: {n} clips | {total_s/3600:.2f} h "
              f"(objetivo {max_hours} h).", flush=True)
        print("[datos] Vuelve a ejecutar la MISMA celda para reanudar desde el "
              "último clip: no se pierde nada.", flush=True)
        return
    _finish_materialization(rows, n, total_s, split, out_dir, t0)


def materialize_split(args, split: str, max_hours: float):
    """Materializa hasta `max_hours` de audio útil en FLAC 16 kHz.

    Prefiere la vía directa por shards parquet (RAM acotada); si el layout del
    repo no encaja o falla el listado, cae al modo streaming de 'datasets'.
    Ambas vías son reanudables (manifiesto) y vigiladas por RAM.
    """
    try:
        shards = _list_parquet_shards(args, split)
    except Exception as e:  # noqa: BLE001
        shards = []
        print(f"[datos] Aviso: no pude listar los parquet del repo ({e}); "
              "uso el modo streaming de datasets.")
    cap_gb = _ram_cap_gb(args)
    if shards:
        _materialize_parquet(args, split, max_hours, shards, cap_gb)
    else:
        _materialize_streaming(args, split, max_hours, cap_gb)


def load_split_local(args, split: str, max_hours: float):
    """Devuelve el split desde la caché local; si no existe, lo materializa."""
    from datasets import load_from_disk

    out_dir = os.path.join(args.data_cache, split)
    if os.path.isdir(out_dir):
        try:
            ds = load_from_disk(out_dir)
            # decode=False: el collator decodifica con soundfile. Así el pipeline
            # no depende de torchcodec (obligatorio para decodificar audio en
            # datasets>=5) ni de la versión de 'datasets' instalada.
            from datasets import Audio
            ds = ds.cast_column("audio", Audio(sampling_rate=16000, decode=False))
            print(f"[datos] Reutilizando caché: {out_dir} "
                  f"({len(ds)} clips, {sum(ds['duration'])/3600:.2f} h)")
            return ds
        except Exception:  # noqa: BLE001
            pass
    materialize_split(args, split, max_hours)
    if not os.path.isdir(out_dir):
        sys.exit(
            f"\n[datos] '{split}' quedó incompleto: la materialización se detuvo "
            "antes de tiempo (límite de RAM, error de red o interrupción).\n"
            "Vuelve a ejecutar la MISMA celda: se reanuda desde el último clip y\n"
            "no se pierde nada de lo ya convertido (manifiesto + FLACs en disco).\n"
        )
    return load_from_disk(out_dir)


# ---------------------------------------------------------------- collator --
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Convierte audio crudo -> log-mel y texto -> tokens dentro del dataloader.

    Así NO se cachean los espectrogramas en disco (ahorra ~30 GB) y la
    extracción se hace en paralelo con los workers del dataloader.
    """

    processor: Any

    @staticmethod
    def _audio_array(field: Any):
        """Devuelve (np.float32 mono, sr) sea cual sea el formato del campo audio.

        Cubre las variantes que aparecen según versión de 'datasets':
        - dict con 'array' ya decodificado (np, o torch si datasets>=5 + torchcodec)
        - dict {bytes, path} SIN decodificar (Audio(decode=False): lo que usa este script)
        - solo 'path'
        Así no dependemos de torchcodec ni de la versión de 'datasets'.
        """
        import io
        import soundfile as sf

        if isinstance(field, dict):
            arr, sr = field.get("array"), field.get("sampling_rate", 16000)
            if arr is None:  # sin decodificar: bytes o path
                if field.get("bytes") is not None:
                    arr, sr = sf.read(io.BytesIO(field["bytes"]), dtype="float32")
                else:
                    arr, sr = sf.read(field["path"], dtype="float32")
        else:
            arr, sr = field, 16000
        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim > 1:  # (canales, muestras) -> mono
            arr = arr.mean(axis=0)
        return arr, sr

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        fe = self.processor.feature_extractor
        tok = self.processor.tokenizer

        # 1) audio -> log-mel de 30 s (el feature extractor de Whisper lo fija)
        input_features = []
        for f in features:
            arr, sr = self._audio_array(f["audio"])
            input_features.append(
                {"input_features": fe(arr, sampling_rate=sr).input_features[0]}
            )
        batch = fe.pad(input_features, padding=True, return_tensors="pt")

        # 2) texto -> input_ids (truncados a 448, el máximo de Whisper)
        label_features = [
            {"input_ids": tok(f["sentence"], truncation=True, max_length=448).input_ids}
            for f in features
        ]
        labels_batch = tok.pad(label_features, padding=True, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 3) quitar el <bos> inicial: el decoder ya lo añade al desplazar labels
        if (labels[:, 0] == tok.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


def make_compute_metrics_whisper(processor):
    """WER sobre las hipótesis generadas vs. las referencias normalizadas."""
    import jiwer

    def compute_metrics(pred):
        pred_ids = np.asarray(pred.predictions)
        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_ids = np.asarray(pred.label_ids).copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        pairs = [(clean_text_es(r), clean_text_es(h)) for r, h in zip(label_str, pred_str)]
        pairs = [(r, h) for r, h in pairs if r]  # ignora referencias vacías
        if not pairs:
            return {"wer": 0.0}
        return {"wer": 100.0 * jiwer.wer([r for r, _ in pairs], [h for _, h in pairs])}

    return compute_metrics


# ------------------------------------------------------------------- main --
def main():
    args = parse_args()
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    # ---------------------------------------------------------------- datos
    # OJO: la carga de datos va ANTES de importar torch/transformers. En modo
    # --prepare_data_only el proceso queda ligero (sin torch ni transformers en
    # RAM: ~1-1.5 GB menos) justo durante la fase más delicada de memoria, p.
    # ej. en Colab free (~12.7 GB de RAM del sistema).
    train_ds = load_split_local(args, "train", args.max_hours)
    eval_ds = load_split_local(args, "validation", args.eval_hours)
    if args.prepare_data_only:
        print("\n[ok] Datos listos. Entrena ahora con:  python whisper_finetune.py")
        return

    import torch
    from transformers import set_seed

    set_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # entradas de forma fija (80x3000): convs más rápidas

    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        print(f"[gpu] {prop.name} | {prop.total_memory / 2**30:.1f} GB de VRAM")
    else:
        print("[gpu] ADVERTENCIA: no se detectó CUDA; en CPU esto sería inviable.")

    # --------------------------------------------------------------- modelo
    from peft import LoraConfig, get_peft_model
    from transformers import (
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
        WhisperFeatureExtractor,
        WhisperForConditionalGeneration,
        WhisperProcessor,
        WhisperTokenizer,
    )

    feature_extractor = WhisperFeatureExtractor.from_pretrained(args.model_id)
    tokenizer = WhisperTokenizer.from_pretrained(
        args.model_id, language=args.language, task="transcribe"
    )
    tokenizer.padding_side = "right"
    processor = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

    # attn_implementation="sdpa": atención fusionada (más rápida y menos VRAM);
    # resultado idéntico, solo cambia el kernel de atención.
    model = WhisperForConditionalGeneration.from_pretrained(
        args.model_id, attn_implementation="sdpa"
    )
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model.config.use_cache = False  # incompatible con gradient checkpointing

    # La evaluación usa model.generate() (predict_with_generate=True): fijamos
    # idioma y tarea para que decodifique en español en vez de depender de la
    # detección automática de idioma clip a clip (inflaría el WER de validación).
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    model.generation_config.language = args.language
    model.generation_config.task = "transcribe"

    # OJO: sin task_type. Con TaskType.SEQ_2_SEQ_LM, PEFT envuelve el modelo en
    # PeftModelForSeq2SeqLM, cuyo forward pasa 'input_ids' explícitamente al
    # modelo base; Whisper no acepta 'input_ids' (es audio-entrada: usa
    # 'input_features') y lanza:
    #   TypeError: WhisperForConditionalGeneration.forward() got an unexpected
    #              keyword argument 'input_ids'
    # Sin task_type, get_peft_model devuelve el PeftModel base, cuyo forward es
    # un passthrough (*args, **kwargs) compatible con Whisper (peft issue #1988).
    # Para LoRA no cambia nada funcional: mismas capas objetivo, mismo entreno.
    lora_cfg = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
    compute_metrics = make_compute_metrics_whisper(processor)

    targs = Seq2SeqTrainingArguments(
        output_dir=args.trainer_dir,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_steps=args.warmup_steps,
        weight_decay=args.weight_decay,
        num_train_epochs=args.epochs,
        lr_scheduler_type="linear",
        fp16=not args.bf16,
        bf16=args.bf16,
        # gradient checkpointing es OPCIONAL (--grad_checkpointing): ahorra VRAM
        # pero recalcula activaciones (~1.3x más cómputo). Sin él el entrenamiento
        # es matemáticamente idéntico, solo más rápido; con LoRA la memoria de
        # optimizador es mínima y en GPUs de 16 GB (T4/L4) sobra sin checkpointing.
        gradient_checkpointing=args.grad_checkpointing,
        gradient_checkpointing_kwargs={"use_reentrant": False}
        if args.grad_checkpointing else None,
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        save_strategy="steps",
        save_steps=args.save_steps,
        save_total_limit=2,
        logging_steps=args.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        remove_unused_columns=False,
        dataloader_num_workers=args.num_workers,
        report_to="none",
        seed=args.seed,
    )

    trainer = Seq2SeqTrainer(
        args=targs,
        model=model,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        processing_class=processor,
    )

    last_ckpt = None
    if args.resume:
        try:
            from transformers.trainer_utils import get_last_checkpoint
            last_ckpt = get_last_checkpoint(args.trainer_dir)
        except Exception:  # noqa: BLE001
            last_ckpt = None
        print(f"[train] Reanudando desde: {last_ckpt}" if last_ckpt
              else "[train] Sin checkpoint previo: empiezo de cero")

    steps_ep = max(1, int(np.ceil(len(train_ds) / (args.batch_size * args.grad_accum))))
    print(f"[train] {len(train_ds)} clips (~{sum(train_ds['duration'])/3600:.1f} h) | "
          f"~{steps_ep} pasos/época x {args.epochs} épocas\n")

    trainer.train(resume_from_checkpoint=last_ckpt)

    metrics = trainer.evaluate()
    wer_final = metrics.get("eval_wer", float("nan"))
    print(f"\n[eval] WER final: {wer_final:.2f} % (sobre {args.eval_hours} h de validación)")

    # --------------------------------------------- exportar modelo fusionado
    print("[export] Fusionando adaptadores LoRA dentro del modelo ...")
    merged = model.merge_and_unload()
    merged.config.use_cache = True
    merged.save_pretrained(args.output_dir)
    processor.save_pretrained(args.output_dir)

    with open(os.path.join(args.output_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(
            f"---\nlanguage: es\nlicense: apache-2.0\nbase_model: {args.model_id}\n"
            f"tags: [whisper, asr, lora, common-voice]\n---\n\n"
            f"# Whisper-small afinado en español (Common Voice + LoRA)\n\n"
            f"- Dataset: {args.dataset_name} ({args.dataset_config}), ~{args.max_hours} h\n"
            f"- Épocas: {args.epochs} | WER en validación (~{args.eval_hours} h): {wer_final:.2f} %\n\n"
            f"## Uso\n\n"
            f"```python\n"
            f"from transformers import pipeline\n"
            f"pipe = pipeline(\"automatic-speech-recognition\", model=\"{args.output_dir}\", "
            f"chunk_length_s=30)\n"
            f"print(pipe(\"audio.wav\")[\"text\"])\n"
            f"```\n"
        )

    print(
        "\n" + "=" * 74 +
        f"\n[OK] Modelo Whisper fusionado y guardado en: {args.output_dir}"
        "\nÚsalo en otro proyecto con:"
        "\n    from transformers import pipeline"
        f"\n    pipe = pipeline('automatic-speech-recognition', model='{args.output_dir}', chunk_length_s=30)"
        "\n    print(pipe('mi_audio.wav')['text'])"
        "\n" + "=" * 74
    )


if __name__ == "__main__":
    main()


## 4. Materializar los datos: 30 h (~10–20 min, una vez por sesión)

Cada 500 clips verás progreso y **uso de RAM**. La celda es **reanudable**:
- Si se corta por RAM/red o la sesión muere, **vuelve a ejecutar esta misma celda**:
  continúa desde el último clip (manifiesto + FLACs en disco), sin repetir nada.
- La celda se corta sola ANTES de agotar la RAM (límite automático del 75 %): verás
  `[aviso] ... conservo TODO lo ya convertido`. Re-ejecútala y seguirá donde quedó.

In [ ]:
!python whisper_finetune.py --prepare_data_only --max_hours 30 --eval_hours 0.5

## 5. Entrenar (~2 h)

- Batch 16, **sin** gradient checkpointing (T4 16 GB: sobra VRAM).
- Eval + checkpoint cada 750 pasos; solo se conservan los 2 últimos checkpoints.
- `--resume`: si la sesión se corta, re-ejecuta las celdas 2→5 y relanza esta;
  continúa desde el último checkpoint de Drive.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python whisper_finetune.py \
  --batch_size 16 --grad_accum 1 --eval_batch_size 16 \
  --epochs 3 --max_hours 30 --eval_hours 0.5 \
  --eval_steps 750 --save_steps 750 --num_workers 2 \
  --trainer_dir "{OUT_BASE}/runs" \
  --output_dir "{OUT_BASE}/modelo" \
  --resume

## 6. Probar el modelo final

⚠️ Solo cuando la celda 5 termine con `[OK] Modelo ... guardado` — antes de eso el modelo no existe (el `[aviso]` de abajo es la protección normal).

In [ ]:
import io, os
import soundfile as sf
from datasets import load_from_disk

MODEL_DIR = f"{OUT_BASE}/modelo"
if not os.path.isdir(MODEL_DIR):
    print(f"[aviso] {MODEL_DIR} aún no existe.")
    print("       Ejecuta primero la celda de entrenamiento (celda 5) y vuelve aquí.")
else:
    from transformers import pipeline
    pipe = pipeline("automatic-speech-recognition", model=MODEL_DIR, chunk_length_s=30)
    ds = load_from_disk("./data/cv_es/validation")
    for i in range(3):
        a = ds[i]["audio"]
        if a.get("array") is not None:
            arr, sr = a["array"], a["sampling_rate"]
        elif a.get("bytes") is not None:
            arr, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
        else:
            arr, sr = sf.read(a["path"], dtype="float32")
        print(f"REF: {ds[i]['sentence']}")
        print(f"HIP: {pipe({'array': arr, 'sampling_rate': sr})['text']}\n")

## Notas

- **Si se desconecta Colab**: el disco efímero (`/content`, incluida la caché de datos) se borra; los checkpoints de Drive no. Re-ejecuta celdas 1→5 y relanza la 6 (`--resume` continúa donde iba).
- **RAM en la celda 5**: la descarga va por el CDN clásico (sin xet, que acumula RAM) y un vigilante corta la celda sola al llegar al 75 % de la RAM, guardando el progreso. Si se corta, **re-ejecuta la celda 5**: se reanuda sin perder nada. Si reincide muchas veces, baja el objetivo (`--max_hours 15`).
- **El modelo final** queda en `Drive/whisper_es/modelo` (~1 GB). Úsalo donde quieras:
  `pipeline("automatic-speech-recognition", model=".../modelo", chunk_length_s=30)`.
- **¿Aún más rápido?** Con 10 h y 4 épocas (`--max_hours 10 --epochs 4` en las celdas 5–6) el entrenamiento baja a ~40–50 min, a costa de ~2–3 puntos más de WER.